# Supervised Machine Learning Pipeline for Image Layer Separation

This notebook decomposes an RGB image into $K$ separate color layers using a U-Net architecture. It includes an automated data pipeline that generates training ground truth using RGB K-Means clustering.

In [ ]:
import os
import glob
import math
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using computational device: {device}')


## 1. Configurations & Error Handling Requirements
We set $K$ as a configurable hyperparameter. We also enforce constraints such as valid $K$ values.

In [ ]:
CONFIG = {
    'K': 4,                    # Number of layers to separate into
    'image_size': (256, 256),  # Resize to avoid Out-Of-Memory (OOM) errors
    'batch_size': 4,
    'epochs': 15,              # Number of training epochs
    'lr': 1e-4,
    'data_dir': './data/images',
    'output_dir': './data/targets',
    'results_dir': './results'
}

# Requirement Validation
assert CONFIG['K'] > 1, 'Error: K must be strictly greater than 1.'

os.makedirs(CONFIG['data_dir'], exist_ok=True)
os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)
print('Directories initialized successfully.')


## 2. Data Pipeline & Ground Truth Generation
Uses K-Means clustering to create target layers. Follows the naming convention `cluster_[index]_[hex_code].png`.

In [ ]:
def generate_ground_truth(image_path, output_base_dir, K):
    if not os.path.exists(image_path):
        raise FileNotFoundError(f'Image not found: {image_path}')
    
    try:
        img = Image.open(image_path).convert('RGB')
        # Automatically resize extremely large images to prevent OOM during K-Means fitting
        img.thumbnail((512, 512))
        img_np = np.array(img)
        H, W, C = img_np.shape
        
        # Flatten for clustering
        pixels = img_np.reshape(-1, 3).astype(np.float32)
        
        kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
        labels = kmeans.fit_predict(pixels)
        centers = kmeans.cluster_centers_
        
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        out_dir = os.path.join(output_base_dir, f'{base_name}_channels')
        os.makedirs(out_dir, exist_ok=True)
        
        target_paths = []
        for k in range(K):
            # Generate binary mask and apply to original image to create the "separated image" target
            mask = (labels == k).reshape(H, W)
            layer = np.zeros_like(img_np)
            layer[mask] = img_np[mask]
            
            # Convert float RGB center to hexadecimal
            hex_code = '%02x%02x%02x' % tuple(map(int, centers[k]))
            filename = f'cluster_{k}_{hex_code}.png'
            filepath = os.path.join(out_dir, filename)
            
            Image.fromarray(layer).save(filepath)
            target_paths.append(filepath)
            
        return out_dir, target_paths
        
    except Exception as e:
        print(f'Error processing {image_path}: {e}')
        return None, []

# Validate image formats. Generate dummy data if the folder is empty.
image_files = glob.glob(os.path.join(CONFIG['data_dir'], '*.[jp][pn]*[g]'))
if not image_files:
    print('No standard images found. Generating a dummy gradient image for demonstration...')
    dummy_path = os.path.join(CONFIG['data_dir'], 'dummy_gradient.png')
    x = np.linspace(0, 255, 256)
    X, Y = np.meshgrid(x, x)
    dummy_img = np.zeros((256, 256, 3), dtype=np.uint8)
    dummy_img[:, :, 0] = X
    dummy_img[:, :, 1] = Y
    dummy_img[:, :, 2] = 128
    Image.fromarray(dummy_img).save(dummy_path)
    image_files = [dummy_path]

print('Clustering inputs and generating ground truth targets...')
dataset_items = []
for img_path in image_files:
    target_dir, _ = generate_ground_truth(img_path, CONFIG['output_dir'], CONFIG['K'])
    if target_dir:
        dataset_items.append({'source': img_path, 'targets': target_dir})
        
print(f'Prepared {len(dataset_items)} image(s) for training.')


## 3. Dataset & DataLoader
Loads the original image alongside its $K$ target layers.

In [ ]:
class LayerSeparationDataset(Dataset):
    def __init__(self, items, K, transform=None):
        self.items = items
        self.K = K
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        source_img = Image.open(item['source']).convert('RGB')
        
        target_files = sorted(glob.glob(os.path.join(item['targets'], 'cluster_*.png')))
        if len(target_files) != self.K:
            raise ValueError('Expected {} targets, found {} in {}'.format(self.K, len(target_files), item.get('targets')))
            
        target_imgs = [Image.open(f).convert('RGB') for f in target_files]
        
        if self.transform:
            source_tensor = self.transform(source_img)
            target_tensors = [self.transform(t) for t in target_imgs]
        else:
            to_tensor = transforms.ToTensor()
            source_tensor = to_tensor(source_img)
            target_tensors = [to_tensor(t) for t in target_imgs]
            
        # Stack into (K*3, H, W) for multi-channel target
        targets_stacked = torch.cat(target_tensors, dim=0)
        return source_tensor, targets_stacked

transform = transforms.Compose([
    transforms.Resize(CONFIG['image_size']),
    transforms.ToTensor()
])

dataset = LayerSeparationDataset(dataset_items, CONFIG['K'], transform=transform)
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True, drop_last=False)


## 4. Model Architecture: U-Net
Encoder-decoder architecture outputting $K \times 3$ channels representing our isolated layers.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=3, features=[64, 128, 256, 512]):
        super().__init__()
        self.downs = nn.ModuleList()
        self.ups = nn.ModuleList()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        for feature in features:
            self.downs.append(DoubleConv(in_channels, feature))
            in_channels = feature

        for feature in reversed(features):
            self.ups.append(nn.ConvTranspose2d(feature*2, feature, kernel_size=2, stride=2))
            self.ups.append(DoubleConv(feature*2, feature))

        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.final_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)
        self.sigmoid = nn.Sigmoid()  # Output pixel boundaries [0, 1]

    def forward(self, x):
        skip_connections = []
        for down in self.downs:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)
        skip_connections = skip_connections[::-1]

        for idx in range(0, len(self.ups), 2):
            x = self.ups[idx](x)
            skip_connection = skip_connections[idx//2]
            
            if x.shape != skip_connection.shape:
                x = TF.resize(x, size=skip_connection.shape[2:])
                
            concat_skip = torch.cat((skip_connection, x), dim=1)
            x = self.ups[idx+1](concat_skip)

        return self.sigmoid(self.final_conv(x))

model = UNet(in_channels=3, out_channels=CONFIG['K'] * 3).to(device)
print('U-Net Architecture instantiated successfully.')


## 5. Loss Functions
Implements a composite loss comprising:
1. **Supervised Loss:** Distance to the K-Means separated target.
2. **Reconstruction Loss:** `MSE(Input, sum(Output_Layers))` enforcing layers sum back to the original without ghosting.
3. **Separation Loss:** Overlapping intensity penalty ensuring cleanly separated color blocks.

In [ ]:
class LayerSeparationLoss(nn.Module):
    def __init__(self, K, lambda_recon=1.0, lambda_sep=0.1, lambda_sup=1.0):
        super().__init__()
        self.K = K
        self.lambda_recon = lambda_recon
        self.lambda_sep = lambda_sep
        self.lambda_sup = lambda_sup
        self.mse = nn.MSELoss()

    def forward(self, pred, target, input_img):
        B, _, H, W = pred.shape
        pred_layers = pred.view(B, self.K, 3, H, W)
        
        # 1. Supervised Loss 
        loss_sup = self.mse(pred, target)
        
        # 2. Reconstruction Loss: ensure summed layers faithfully recreate input
        summed_layers = torch.sum(pred_layers, dim=1)
        loss_recon = self.mse(summed_layers, input_img)
        
        # 3. Separation Loss: Penalize overlapping channel energies
        layer_energies = torch.norm(pred_layers, dim=2) # (B, K, H, W)
        loss_sep = 0.0
        if self.K > 1:
            for i in range(self.K):
                for j in range(i+1, self.K):
                    overlap = layer_energies[:, i] * layer_energies[:, j]
                    loss_sep += overlap.mean()
            loss_sep = loss_sep / (self.K * (self.K - 1) / 2)
            
        total_loss = (self.lambda_sup * loss_sup) + (self.lambda_recon * loss_recon) + (self.lambda_sep * loss_sep)
        return total_loss, loss_sup, loss_recon, loss_sep

criterion = LayerSeparationLoss(K=CONFIG['K'])
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])


## 6. Training Loop

In [ ]:
print('Starting Training Pipeline...')
history = {'total': [], 'sup': [], 'recon': [], 'sep': []}

for epoch in range(CONFIG['epochs']):
    model.train()
    epoch_loss, epoch_sup, epoch_recon, epoch_sep = 0.0, 0.0, 0.0, 0.0
    
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss, l_sup, l_recon, l_sep = criterion(outputs, targets, inputs)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        epoch_sup += l_sup.item()
        epoch_recon += l_recon.item()
        epoch_sep += l_sep.item() if isinstance(l_sep, torch.Tensor) else l_sep
        
    num_batches = len(dataloader)
    if num_batches > 0:
        history['total'].append(epoch_loss / num_batches)
        history['sup'].append(epoch_sup / num_batches)
        history['recon'].append(epoch_recon / num_batches)
        history['sep'].append(epoch_sep / num_batches)
    
    print('Epoch [{}/{}] Total Loss: {:.4f} | Sup: {:.4f} | Recon: {:.4f} | Sep: {:.4f}'.format(
        epoch+1, CONFIG['epochs'], 
        history['total'][-1], history['sup'][-1], history['recon'][-1], history['sep'][-1]
    ))

print('Training Complete.')


## 7. Inference & Visualization
Tests the model on an image and saves the separated component files to disk.

In [ ]:
def visualize_results(model, dataset, idx=0):
    if len(dataset) == 0: return
    
    model.eval()
    with torch.no_grad():
        input_tensor, _ = dataset[idx]
        input_batch = input_tensor.unsqueeze(0).to(device)
        output = model(input_batch).squeeze(0).cpu()
        output_layers = output.view(CONFIG['K'], 3, output.shape[1], output.shape[2])
        
    input_img = input_tensor.permute(1, 2, 0).numpy()
    
    fig, axes = plt.subplots(1, CONFIG['K'] + 2, figsize=(3 * (CONFIG['K'] + 2), 3))
    axes[0].imshow(np.clip(input_img, 0, 1))
    axes[0].set_title('Input Image')
    axes[0].axis('off')
    
    summed_output = np.zeros_like(input_img)
    for k in range(CONFIG['K']):
        layer_img = output_layers[k].permute(1, 2, 0).numpy()
        axes[k+1].imshow(np.clip(layer_img, 0, 1))
        axes[k+1].set_title(f'Layer {k+1}')
        axes[k+1].axis('off')
        
        summed_output += layer_img
        
        # Save predictions for verification step
        out_img_pil = Image.fromarray((np.clip(layer_img, 0, 1) * 255).astype(np.uint8))
        out_img_pil.save(os.path.join(CONFIG['results_dir'], f'pred_layer_{k}.png'))
        
    axes[-1].imshow(np.clip(summed_output, 0, 1))
    axes[-1].set_title('Summed Layers')
    axes[-1].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_results(model, dataset, idx=0)


## 8. Requirements Verification (PSNR)
Loads the PNGs strictly from disk, mathematically adds them, and validates against the input image to confirm no ghosting or gaps are present.

In [ ]:
def verify_reconstruction():
    if len(dataset) == 0:
        print('No data to verify.')
        return
        
    input_tensor, _ = dataset[0]
    orig_img = (input_tensor.permute(1, 2, 0).numpy() * 255).astype(np.float32)
    
    summed_img = np.zeros_like(orig_img)
    for k in range(CONFIG['K']):
        layer_path = os.path.join(CONFIG['results_dir'], f'pred_layer_{k}.png')
        if os.path.exists(layer_path):
            # Load from saved PNG image
            layer_img = np.array(Image.open(layer_path)).astype(np.float32)
            summed_img += layer_img
        else:
            print(f'Warning: Missing prediction layer at {layer_path}')
            
    summed_img = np.clip(summed_img, 0, 255)
    
    mse = np.mean((orig_img - summed_img) ** 2)
    psnr = float('inf') if mse == 0 else 20 * math.log10(255.0 / math.sqrt(mse))
    
    print('--- Verification Results ---')
    print(f'Original Image Shape: {orig_img.shape}')
    print(f'Summed Image Shape: {summed_img.shape}')
    print(f'Calculated MSE: {mse:.4f}')
    print(f'Calculated PSNR: {psnr:.2f} dB')
    
    if psnr > 30:
        print('\n✅ SUCCESS: High quality reconstruction (PSNR > 30 dB). Layers combine cleanly without ghosting.')
    else:
        print('\n⚠️ WARNING: Visible differences detected. The model may need more epochs or tuning for complex scenes.')

verify_reconstruction()
